# Data Collection on NA's challenger league (top 300 players)

Pulls the **most recent 10 ranked solo/duo matches** (`MATCHES_PER_PLAYER` in `src/config.py`) for the **top 100 NA players** by LP.

| Step | Endpoint | Routing |
|---|---|---|
| 1. Top 300 players | `league-v4 /challengerleagues/by-queue/RANKED_SOLO_5x5` | `na1` |
| 2. Match IDs per player | `match-v5 /matches/by-puuid/{puuid}/ids` | `americas` |
| 3. Match details | `match-v5 /matches/{matchId}` | `americas` |

> **Runtime:** 1 + 100 + up to 1,000 requests (one per unique match). A dev key allows 100 requests / 2 min, so this takes **~20 min**.  
> Every response is cached to `data/raw/`, so the notebook can be stopped and re-run safely; it resumes where it left off.

## Setup

In [2]:
%%capture

pip install -r requirements.txt

In [1]:
%load_ext autoreload
%autoreload 2

import sys, logging
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

sys.path.append(str(Path.cwd().parent))  # make `src` importable from notebooks/
from src.config import TOP_N, MATCHES_PER_PLAYER, PROCESSED_DIR
from src.riot_api import get_challenger_league, get_match_ids, get_match

logging.basicConfig(level=logging.WARNING, format="%(levelname)s | %(message)s")

C:\Users\Ben\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Top 300 Players
The Challenger ladder holds the top 300 NA players.
`refresh=True` pulls a fresh ladder instead of using the cached one.

In [2]:
league = get_challenger_league(refresh=False)

players = (
    pd.DataFrame(league["entries"])
      .sort_values("leaguePoints", ascending=False)
      .head(TOP_N)
      .reset_index(drop=True)
)
players.insert(0, "ladder_rank", players.index + 1)
players["games"] = players["wins"] + players["losses"]
players["win_rate"] = players["wins"] / players["games"]

print(f"{len(players)} players | LP range: {players.leaguePoints.min()}–{players.leaguePoints.max()}")
players.head()

300 players | LP range: 1765–4041


,ladder_rank,puuid,leaguePoints,rank,wins,losses,veteran,inactive,freshBlood,hotStreak,games,win_rate
0,1,QbUo_rUH-xUixdyYxetIJf0C6QIOA_tXBWmrOkRo1_B8iV...,4041,I,358,245,True,False,False,False,603,0.593698
1,2,8WfFV8TPkd6DIaajOHARJAJR8BJF8U9sTmFRzt4SfhmSe7...,4013,I,558,352,True,False,False,True,910,0.613187
2,3,lmPjvRxc2s1Ha9Wbb8y6SkcbvZUF1vN86QNM_Ib4cOQMRt...,3523,I,636,504,True,False,False,True,1140,0.557895
3,4,XwiJ6fkuW_8DC9PDNvlKYAFlgm4e6j952DE0TqJWWt9Rxl...,3228,I,698,545,True,False,False,False,1243,0.561545
4,5,i627oHfp0rlM_VbmJMFze12mD6UYaiao9Hl6XTmYHKrSON...,3217,I,497,421,True,False,False,False,918,0.541394


## 2. Recent Match IDs
One request per player for their last `MATCHES_PER_PLAYER` ranked solo matches (`queue=420`).  
Top players often share games, so match IDs are de-duplicated before fetching details.

In [3]:
match_ids = {}   # puuid -> list of match IDs
failed_players = []

for puuid in tqdm(players["puuid"], desc="Match IDs"):
    try:
        match_ids[puuid] = get_match_ids(puuid, count=MATCHES_PER_PLAYER)
    except (RuntimeError, PermissionError) as e:
        logging.error("Skipping %s: %s", puuid, e)
        failed_players.append(puuid)

unique_ids = sorted({m for ids in match_ids.values() for m in ids})
total_ids = sum(len(ids) for ids in match_ids.values())
print(f"{total_ids} player-matches -> {len(unique_ids)} unique matches | failed players: {len(failed_players)}")

Match IDs: 100%|██████████| 300/300 [04:00<00:00,  1.24it/s]

30000 player-matches -> 14367 unique matches | failed players: 0


## 3. Match Details
Fetch each unique match once. Failures are logged and skipped so one bad request doesn't stop the run.  
Re-run this cell to retry any `failed_matches`.

In [4]:
matches = {}
failed_matches = []

for mid in tqdm(unique_ids, desc="Matches"):
    try:
        data = get_match(mid)
        if data is not None:
            matches[mid] = data
    except PermissionError:
        raise  # expired key: stop and regenerate
    except RuntimeError as e:
        logging.error("Skipping %s: %s", mid, e)
        failed_matches.append(mid)

print(f"Fetched {len(matches)}/{len(unique_ids)} matches | failed: {len(failed_matches)}")

Matches:  17%|█▋        | 2499/14367 [48:23<44:26,  4.45it/s]    WARNING | Network error (('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))), attempt 1
WARNING | 429 rate limited; sleeping 6s
Matches: 100%|██████████| 14367/14367 [4:48:59<00:00,  1.21s/it]   

Fetched 14367/14367 matches | failed: 0


## 4. Build Player-Match Table
One row per **(top-300 player, match)**, built from that player's entry in the match's `participants`.  
Remakes (early surrender / under 5 min) are flagged, not dropped.

In [5]:
STAT_COLS = [
    "riotIdGameName", "riotIdTagline", "championName", "teamPosition", "teamId", "win",
    "kills", "deaths", "assists", "totalMinionsKilled", "neutralMinionsKilled",
    "goldEarned", "totalDamageDealtToChampions", "damageDealtToObjectives",
    "visionScore", "wardsPlaced", "wardsKilled", "champLevel", "timeCCingOthers",
    "firstBloodKill", "turretKills", "gameEndedInEarlySurrender",
]

def player_match_row(puuid: str, match: dict) -> dict | None:
    """Extract one player's stats + match metadata from a match-v5 payload."""
    info = match["info"]
    p = next((x for x in info["participants"] if x["puuid"] == puuid), None)
    if p is None:
        return None
    row = {
        "matchId": match["metadata"]["matchId"],
        "puuid": puuid,
        "gameCreation": pd.to_datetime(info["gameCreation"], unit="ms"),
        "gameDuration_min": info["gameDuration"] / 60,
        "gameVersion": info["gameVersion"],
        "patch": ".".join(info["gameVersion"].split(".")[:2]),
    }
    row.update({c: p.get(c) for c in STAT_COLS})
    return row

rows = [
    r for puuid, ids in match_ids.items() for mid in ids
    if mid in matches and (r := player_match_row(puuid, matches[mid])) is not None
]
df = pd.DataFrame(rows)

# Derived features
df["cs"] = df["totalMinionsKilled"] + df["neutralMinionsKilled"]
df["cs_per_min"] = df["cs"] / df["gameDuration_min"]
df["gold_per_min"] = df["goldEarned"] / df["gameDuration_min"]
df["kda"] = (df["kills"] + df["assists"]) / df["deaths"].clip(lower=1)
df["is_remake"] = df["gameEndedInEarlySurrender"] | (df["gameDuration_min"] < 5)

print(df.shape)
df.head()

(30000, 33)


,matchId,puuid,gameCreation,gameDuration_min,gameVersion,patch,riotIdGameName,riotIdTagline,championName,teamPosition,...,champLevel,timeCCingOthers,firstBloodKill,turretKills,gameEndedInEarlySurrender,cs,cs_per_min,gold_per_min,kda,is_remake
0,NA1_5659998027,QbUo_rUH-xUixdyYxetIJf0C6QIOA_tXBWmrOkRo1_B8iV...,2026-10-14 03:55:52.729,17.866667,16.20.824.8524,16.20,C9 Loki,kr3,Akali,MIDDLE,...,12,3,False,0,False,163,9.123134,384.738806,1.000000,False
1,NA1_5659953897,QbUo_rUH-xUixdyYxetIJf0C6QIOA_tXBWmrOkRo1_B8iV...,2026-10-14 03:01:47.670,32.816667,16.20.824.8524,16.20,C9 Loki,kr3,Pyke,UTILITY,...,13,33,False,0,False,32,0.975114,311.000508,2.333333,False
2,NA1_5659936919,QbUo_rUH-xUixdyYxetIJf0C6QIOA_tXBWmrOkRo1_B8iV...,2026-10-14 02:34:36.093,23.583333,16.20.824.8524,16.20,C9 Loki,kr3,Lissandra,MIDDLE,...,15,32,False,0,False,218,9.243816,430.303887,2.400000,False
3,NA1_5659295272,QbUo_rUH-xUixdyYxetIJf0C6QIOA_tXBWmrOkRo1_B8iV...,2026-10-13 01:39:45.362,31.933333,16.20.824.8524,16.20,C9 Loki,kr3,Ahri,MIDDLE,...,16,25,False,1,False,267,8.361169,411.638831,1.857143,False
4,NA1_5659128611,QbUo_rUH-xUixdyYxetIJf0C6QIOA_tXBWmrOkRo1_B8iV...,2026-10-12 21:38:46.451,26.216667,16.20.824.8524,16.20,C9 Loki,kr3,Akali,MIDDLE,...,17,4,True,3,False,236,9.001907,629.218055,8.666667,False


### Add Riot IDs to the player table
The league endpoint doesn't return names, so take each player's most recent Riot ID from their matches.

In [6]:
latest_names = (
    df.sort_values("gameCreation")
      .groupby("puuid")[["riotIdGameName", "riotIdTagline"]]
      .last()
)
players = players.merge(latest_names, on="puuid", how="left")
players[["ladder_rank", "riotIdGameName", "leaguePoints", "wins", "losses", "win_rate"]].head(10)

,ladder_rank,riotIdGameName,leaguePoints,wins,losses,win_rate
0,1,C9 Loki,4041,358,245,0.593698
1,2,never type,4013,558,352,0.613187
2,3,Lobby Admin,3523,636,504,0.557895
3,4,Davemon,3228,698,545,0.561545
4,5,LFT Sheiden,3217,497,421,0.541394
5,6,Samuel Broadley,3180,419,280,0.599428
6,7,CHICKENORBUNNY,3159,338,253,0.571912
7,8,Zven,3134,837,699,0.544922
8,9,cant type,3118,514,344,0.599068
9,10,PhyMini,3113,508,417,0.549189


## Save Data

In [ ]:
print("Matches per player:\n", df.groupby("puuid").size().describe().round(1), sep="")
print(f"\nRemakes: {df.is_remake.sum()} | Date range: {df.gameCreation.min():%Y-%m-%d} -> {df.gameCreation.max():%Y-%m-%d}")
print(f"Patches: {sorted(df.patch.unique())}")

players.to_csv(PROCESSED_DIR / "top300_players.csv")
df.to_csv(PROCESSED_DIR / "player_matches.csv")
print(f"\nSaved to {PROCESSED_DIR}")

Matches per player:
count    300.0
mean     100.0
std        0.0
min      100.0
25%      100.0
50%      100.0
75%      100.0
max      100.0
dtype: float64

Remakes: 281 | Date range: 2026-07-01 -> 2026-10-15
Patches: ['16.13', '16.14', '16.15', '16.16', '16.17', '16.18', '16.19', '16.20']

Saved to C:\Users\Ben\Documents\Data Science Projects\Study-of-NA-s-top-100-LoL-players\data\processed
